# Sweep 14b — Graph Layer Types (5-context matrix)

**Question:** Which GNN layer type is best for the drug graph, and does it depend on modalities?

**Design:** 6 layer types × 5 contexts × 1 seed (discovery) = 30 runs ~5h
then top-2 per context × 3 seeds (confirmation) ~3h

All runs use `graph_encoder_type=drug_gnn` with `graph_layer_type` swapped.
Layers: hgt (baseline), gat, han, gcn, sage, digcn

**Locked-in:** aggregator=last, ar_seq_len=20, note_proj=128, encoder=transformer, fusion=film

In [ ]:
import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
!pip install -q torch_geometric
!pip install -q pyyaml pandas numpy scikit-learn


In [ ]:
import os, sys, glob

if os.path.exists("/kaggle"):
    print("Running on Kaggle")
    os.chdir("/kaggle/working")
    os.system("rm -rf ./data ./src")
    os.makedirs("data/processed", exist_ok=True)
    os.makedirs("data/embeddings", exist_ok=True)

    train_paths = glob.glob("/kaggle/input/**/train.py", recursive=True)
    if not train_paths:
        raise FileNotFoundError("train.py not found in /kaggle/input")
    src_dir = os.path.dirname(train_paths[0])
    os.system(f"cp -r {src_dir} /kaggle/working/src")
    sys.path.append("/kaggle/working/src")

    for find_file in ["cohort_mimic3.pkl", "records_final.pkl"]:
        found = glob.glob(f"/kaggle/input/**/{find_file}", recursive=True)
        if found:
            processed_dir = os.path.dirname(found[0])
            print(f"Found processed dir at: {processed_dir}")
            for root, dirs, files in os.walk(processed_dir):
                rel = os.path.relpath(root, processed_dir)
                target_dir = os.path.join("./data/processed", rel) if rel != "." else "./data/processed"
                os.makedirs(target_dir, exist_ok=True)
                for fname in files:
                    src = os.path.join(root, fname)
                    link = os.path.join(target_dir, fname)
                    if not os.path.exists(link):
                        try:
                            os.symlink(src, link)
                        except OSError:
                            os.system(f"cp '{src}' '{link}'")
            break

    emb_paths = glob.glob("/kaggle/input/**/code_embeddings.pt", recursive=True)
    if not emb_paths:
        raise FileNotFoundError("code_embeddings.pt not found")
    emb_dir = os.path.dirname(emb_paths[0])
    for fpath in glob.glob(f"{emb_dir}/*"):
        fname = os.path.basename(fpath)
        link = f"./data/embeddings/{fname}"
        if not os.path.exists(link):
            os.symlink(fpath, link)

    note_paths = glob.glob("/kaggle/input/**/note_embeddings_mimic3.pkl", recursive=True)
    if note_paths:
        note_src = note_paths[0]
        note_link = "./data/processed/note_embeddings_mimic3.pkl"
        if not os.path.exists(note_link):
            os.symlink(note_src, note_link)
        print(f"Note embeddings linked: {note_src}")
    else:
        print("WARNING: note_embeddings_mimic3.pkl not found — notes contexts will fail")

    reg_path = "/kaggle/working/src/model/registry.py"
    if os.path.exists(reg_path):
        with open(reg_path, "r") as rf:
            content = rf.read()
        if "import inspect" not in content:
            old = "        return cls(**kwargs)"
            new = (
                "        import inspect\n"
                "        sig = inspect.signature(cls.__init__)\n"
                "        has_var_keyword = any(p.kind == inspect.Parameter.VAR_KEYWORD "
                "for p in sig.parameters.values())\n"
                "        if has_var_keyword:\n"
                "            filtered_kwargs = kwargs\n"
                "        else:\n"
                "            filtered_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}\n"
                "        return cls(**filtered_kwargs)"
            )
            if old in content:
                content = content.replace(old, new)
                with open(reg_path, "w") as wf:
                    wf.write(content)
                print("  registry.py patched")

    # Patch DrugGNN to accept ehr_adj (model.py passes it to all graph encoders;
    # only hgt_ehr_feat uses it — DrugGNN must absorb and ignore it).
    gnn_path = "/kaggle/working/src/model/graph_encoders/drug_gnn.py"
    if os.path.exists(gnn_path):
        with open(gnn_path, "r") as rf:
            content = rf.read()
        if "ehr_adj" not in content:
            old = "        use_lab_nodes: bool = False,\n        lab_embeddings: torch.Tensor | None = None,   # (num_labs, 768)\n    ):"
            new = "        use_lab_nodes: bool = False,\n        lab_embeddings: torch.Tensor | None = None,   # (num_labs, 768)\n        ehr_adj=None,  # absorbed — only hgt_ehr_feat uses this\n        **kwargs,\n    ):"
            if old in content:
                content = content.replace(old, new)
                with open(gnn_path, "w") as wf:
                    wf.write(content)
                print("  drug_gnn.py patched (ehr_adj absorbed)")
            else:
                print("  WARNING: drug_gnn.py patch target not found — check manually")

    # Patch DCMAScorer: add nn.Linear projections so notes_repr (768d) and
    # labs_repr (400d) are both projected to hidden_dim before torch.cat(dim=1).
    # Uses regular Linear (not LazyLinear) so parameter counting at init is safe.
    dcma_path = "/kaggle/working/src/model/decoders/dcma_decoder.py"
    if os.path.exists(dcma_path):
        with open(dcma_path, "r") as rf:
            content = rf.read()
        if "note_proj" not in content:
            content = content.replace(
                "    def __init__(self, hidden_dim: int = 256, dropout: float = 0.3, **kwargs):\n"
                "        super().__init__()\n"
                "        self.attn = DrugModalityAttention(hidden_dim, dropout)",
                "    def __init__(self, hidden_dim: int = 256, dropout: float = 0.3,\n"
                "                 note_repr_dim: int = 768, lab_repr_dim: int = 400, **kwargs):\n"
                "        super().__init__()\n"
                "        self.attn = DrugModalityAttention(hidden_dim, dropout)\n"
                "        self.hidden_dim = hidden_dim\n"
                "        self.note_proj = nn.Linear(note_repr_dim, hidden_dim)\n"
                "        self.lab_proj  = nn.Linear(lab_repr_dim,  hidden_dim)",
            )
            content = content.replace(
                "            tokens.append(F.normalize(notes_repr, dim=-1).unsqueeze(1))",
                "            tokens.append(self.note_proj(F.normalize(notes_repr, dim=-1)).unsqueeze(1))",
            )
            content = content.replace(
                "            tokens.append(F.normalize(labs_repr, dim=-1).unsqueeze(1))",
                "            tokens.append(self.lab_proj(F.normalize(labs_repr, dim=-1)).unsqueeze(1))",
            )
            with open(dcma_path, "w") as wf:
                wf.write(content)
            print("  dcma_decoder.py patched (note_proj + lab_proj added)")
        else:
            print("  dcma_decoder.py already patched")

print("Setup done:", os.getcwd())


In [ ]:
import yaml, subprocess, gc, json, numpy as np
from pathlib import Path

BASE_CONFIG = "src/config.yaml"

def make_config(output_path, model_overrides=None, preprocessing_overrides=None, smoke_epochs=None):
    with open(BASE_CONFIG, "r") as f:
        cfg = yaml.safe_load(f)
    if model_overrides:
        for k, v in model_overrides.items():
            cfg["model"][k] = v
    if preprocessing_overrides:
        for k, v in preprocessing_overrides.items():
            cfg["preprocessing"][k] = v
    if smoke_epochs is not None:
        cfg["training"]["epochs"] = smoke_epochs
    with open(output_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    return str(output_path)


# ── Locked-in best config (sweeps 11a-13a) ────────────────────────────────
BEST_MODEL = {
    "ar_max_seq_len": 20,
    "note_proj_dim": 128,
    "graph_encoder_type": "drug_gnn",
    "graph_layer_type": "hgt",
    "hgt_layers": 2,
}
BACKBONE_FLAGS = (
    "--device cuda "
    "--visit_level_training "
    "--fusion_strategy film "
    "--aggregator_type last "
)
LAB_FLAGS = "--lab_pkl data/processed/lab_vectors_200labs.pkl --num_labs 200"

# 5 modality contexts
CONTEXTS = {
    "naked":      {"model_ov": {"copy_mechanism": False},
                   "prep_ov":  {"note_method": "none", "lab_dim": 0},  "flags": ""},
    "notes_only": {"model_ov": {"copy_mechanism": False},
                   "prep_ov":  {"lab_dim": 0},                          "flags": ""},
    "labs_only":  {"model_ov": {"copy_mechanism": False},
                   "prep_ov":  {"note_method": "none", "lab_dim": 400}, "flags": LAB_FLAGS},
    "notes_labs": {"model_ov": {"copy_mechanism": False},
                   "prep_ov":  {"lab_dim": 400},                        "flags": LAB_FLAGS},
    "full":       {"model_ov": {},
                   "prep_ov":  {"lab_dim": 400},                        "flags": LAB_FLAGS},
}
DISC_SEED = 42

GRAPH_LAYERS = [
    ("hgt",   "HGT — Heterogeneous Graph Transformer (baseline)"),
    ("gat",   "GAT — Graph Attention Network"),
    ("han",   "HAN — Heterogeneous Attention Network"),
    ("gcn",   "GCN — Graph Convolutional Network"),
    ("sage",  "GraphSAGE — inductive neighbourhood aggregation"),
    ("digcn", "DiGCN — directed graph convolution"),
]
print(f"Ready: {len(GRAPH_LAYERS)} graph layers x {len(CONTEXTS)} contexts")

SEEDS = [42, 123, 456]
reports_dir = Path("experiment_reports/sweep_14b_graph_layers")
results_log = []

def run_group(configs, group_tag):
    import torch
    for name, cfg_path, extra_flags in configs:
        for seed in SEEDS:
            run_name = f"{name}_seed{seed}"
            run_dir  = reports_dir / run_name
            run_dir.mkdir(parents=True, exist_ok=True)
            if list(run_dir.glob("result_*.json")):
                print(f"  [SKIP] {run_name} — result already exists")
                results_log.append(f"SKIP: {run_name}")
                continue
            log_path = run_dir / "training_log.txt"
            cmd = (
                f"python -u src/train.py --config {cfg_path} "
                f"{BACKBONE_FLAGS} {extra_flags} "
                f"--seed {seed} --results_dir {run_dir}"
            )
            print(f"\n=== [{group_tag}] {run_name} ===")
            try:
                with open(log_path, "w") as lf:
                    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                            stderr=subprocess.STDOUT, text=True)
                    for line in proc.stdout:
                        print(line, end="")
                        lf.write(line)
                    proc.wait()
                status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
                results_log.append(f"{status}: {run_name}")
            except Exception as e:
                results_log.append(f"CRASH: {run_name}: {e}")
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"Config helpers ready. Reports -> {reports_dir}")


In [ ]:
# ── Smoke test: 1 epoch, 3 configs ───────────────────────────────────────────
Path("smoke_s14b").mkdir(exist_ok=True)
SMOKE_CASES = [("hgt", "naked"), ("gat", "full"), ("gcn", "labs_only")]
smoke_results = []
for layer_key, ctx_name in SMOKE_CASES:
    ctx = CONTEXTS[ctx_name]
    cfg_path = f"s14b_smoke_{layer_key}_{ctx_name}.yaml"
    make_config(cfg_path,
                model_overrides={**BEST_MODEL, **ctx["model_ov"],
                                 "graph_layer_type": layer_key,
                                 "graph_encoder_type": "drug_gnn"},
                preprocessing_overrides=ctx["prep_ov"], smoke_epochs=1)
    cmd = (f"python -u src/train.py --config {cfg_path} "
           f"{BACKBONE_FLAGS} {ctx['flags']} "
           f"--seed {DISC_SEED} --results_dir smoke_s14b/{layer_key}_{ctx_name}")
    print(f"SMOKE {layer_key}/{ctx_name}: {cmd}")
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    for ln in (proc.stdout + proc.stderr).strip().split("\n")[-5:]: print(ln)
    status = "PASS" if proc.returncode == 0 else f"FAIL ({proc.returncode})"
    smoke_results.append(f"{status}: {layer_key}/{ctx_name}")
    print(f">>> {status}\n")
for r in smoke_results: print(r)
if any("FAIL" in r for r in smoke_results):
    raise RuntimeError("Smoke tests failed.")
print("All smoke tests passed.")


In [ ]:
# ── Discovery — 6 layers × 5 contexts × seed42 = 30 runs ─────────────────────
print("\n" + "#"*70 + "\n# DISCOVERY TIER\n" + "#"*70)
disc_results = {}
disc_configs = []
for layer_key, layer_note in GRAPH_LAYERS:
    print(f"\n  Layer: {layer_key} — {layer_note}")
    for ctx_name, ctx in CONTEXTS.items():
        cfg_name = f"{layer_key}_{ctx_name}"
        cfg_path = f"s14b_{cfg_name}.yaml"
        make_config(cfg_path,
                    model_overrides={**BEST_MODEL, **ctx["model_ov"],
                                     "graph_layer_type": layer_key,
                                     "graph_encoder_type": "drug_gnn"},
                    preprocessing_overrides=ctx["prep_ov"])
        disc_configs.append((cfg_name, cfg_path, ctx["flags"]))

run_group(disc_configs, "14b")
for name, _, _ in disc_configs:
    rfs = list((reports_dir / f"{name}_seed{DISC_SEED}").glob("result_*.json"))
    if rfs:
        dd = json.load(open(rfs[-1]))
        sec = dd.get("test_metrics", {})
        disc_results[name] = float(sec.get("Jaccard", sec.get("jaccard", 0.0)))
print("\nDiscovery done.")


In [ ]:
# ── Discovery summary + top-2 per context ────────────────────────────────────
print("\nDISCOVERY RESULTS (seed=42)")
print(f"  {'Layer':<10}" + "".join(f"  {c:>12}" for c in CONTEXTS))
print("  " + "-"*75)
top2_per_ctx = {ctx: [] for ctx in CONTEXTS}
for layer_key, _ in GRAPH_LAYERS:
    row = f"  {layer_key:<10}"
    for ctx_name in CONTEXTS:
        jac = disc_results.get(f"{layer_key}_{ctx_name}", 0.0)
        row += f"  {jac:>12.4f}"
        top2_per_ctx[ctx_name].append((jac, layer_key))
    print(row)

print("\nTop-2 per context for confirmation:")
confirmation_configs = []
for ctx_name, scores in top2_per_ctx.items():
    scores.sort(reverse=True)
    top2 = scores[:2]
    print(f"  {ctx_name:12s}: {[e for _,e in top2]}  Jac: {[f'{j:.4f}' for j,_ in top2]}")
    for _, layer_key in top2:
        confirmation_configs.append((layer_key, ctx_name))


In [ ]:
# ── Confirmation tier — top-2 per context × seeds 123 + 456 ──────────────────
layer_ov_map = {k: {"graph_layer_type": k, "graph_encoder_type": "drug_gnn"}
                for k, _ in GRAPH_LAYERS}
print(f"\nConfirmation: {len(confirmation_configs)} configs × 2 new seeds")
for layer_key, ctx_name in confirmation_configs:
    ctx = CONTEXTS[ctx_name]
    cfg_name = f"{layer_key}_{ctx_name}"
    cfg_path = f"s14b_{cfg_name}.yaml"
    make_config(cfg_path,
                model_overrides={**BEST_MODEL, **ctx["model_ov"], **layer_ov_map[layer_key]},
                preprocessing_overrides=ctx["prep_ov"])
    for seed in [123, 456]:
        run_dir = reports_dir / f"{cfg_name}_seed{seed}"
        run_dir.mkdir(parents=True, exist_ok=True)
        cmd = (f"python -u src/train.py --config {cfg_path} "
               f"{BACKBONE_FLAGS} {ctx['flags']} "
               f"--seed {seed} --results_dir {run_dir}")
        print(f"\n=== [14b-conf] {run_dir.name} ===")
        try:
            with open(run_dir / "training_log.txt", "w") as lf:
                proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True)
                for line in proc.stdout: print(line, end=""); lf.write(line)
                proc.wait()
            status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
            results_log.append(f"{status}: {run_dir.name}")
        except Exception as e:
            results_log.append(f"CRASH: {run_dir.name}: {e}")
        gc.collect()
        import torch
        if torch.cuda.is_available(): torch.cuda.empty_cache()
print("\nConfirmation done.")


In [ ]:
import json, numpy as np
from pathlib import Path

def get_metric(d, key):
    sec = d.get("test_metrics", {})
    for k in [key, key.replace("_", " "), key.title()]:
        if k in sec: return float(sec[k])
        if k in d:   return float(d[k])
    return 0.0

all_results = {}
for jp in sorted(reports_dir.rglob("result_*.json")):
    with open(jp) as f: d = json.load(f)
    run_name = jp.parent.name
    idx = run_name.rfind("_seed")
    if idx == -1: continue
    all_results.setdefault(run_name[:idx], []).append(d)

def summarize(name):
    runs = all_results.get(name, [])
    if not runs: return None
    jac  = [get_metric(d, "jaccard")  for d in runs]
    f1   = [get_metric(d, "f1")       for d in runs]
    ddi  = [get_metric(d, "DDI Rate") for d in runs]
    pr   = [get_metric(d, "PRAUC")    for d in runs]
    return dict(jac=np.mean(jac), std=np.std(jac, ddof=1) if len(jac)>1 else 0,
                f1=np.mean(f1), ddi=np.mean(ddi), prauc=np.mean(pr), n=len(jac))

def print_table(header, rows):
    print(f"\n  {header}")
    print(f"  {'Config':<30} {'Jac':>7} {'std':>6}  {'F1':>6}  {'DDI':>6}  {'PRAUC':>7}  n")
    print("  " + "-"*75)
    sums = [(lbl, k, summarize(k)) for lbl, k in rows]
    valid = [s for _,_,s in sums if s]
    best = max(s["jac"] for s in valid) if valid else 0
    for lbl, k, s in sums:
        if s is None:
            print(f"  {lbl:<30} (missing)")
            continue
        mark = " *" if abs(s["jac"] - best) < 1e-6 else ""
        print(f"  {lbl:<30} {s['jac']:.4f} {s['std']:.4f}  {s['f1']:.4f}  "
              f"{s['ddi']:.4f}  {s['prauc']:.4f}  {s['n']}{mark}")

print("\n" + "="*80)

print_table("GRAPH LAYER RESULTS — by context",
    [(f"{l}_{c}", f"{l}_{c}") for l, _ in GRAPH_LAYERS for c in CONTEXTS])

print("="*80)


In [ ]:
total   = len(results_log)
success = sum(1 for r in results_log if "SUCCESS" in r)
failed  = sum(1 for r in results_log if "FAILED"  in r)
crashed = sum(1 for r in results_log if "CRASH"   in r)
print(f"\nRun log: {total} total | {success} success | {failed} failed | {crashed} crashed")
if failed + crashed > 0:
    print("\nFailed/crashed runs:")
    for r in results_log:
        if "SUCCESS" not in r: print(f"  {r}")


In [ ]:
import zipfile
from pathlib import Path

zip_name = "reports_sweep_14b_graph_layers.zip"
rd = reports_dir
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(rd.rglob("result_*.json")):
        zf.write(p, p.relative_to(rd.parent))
    for p in sorted(rd.rglob("training_log.txt")):
        zf.write(p, p.relative_to(rd.parent))
print(f"Exported -> {zip_name}")
